# 17 — Direct Preference Optimization (DPO)

**Network LLM Engineering — Part IV — Post-Training**

### Learning goals
- Understand chosen/rejected preference training
- Prepare networking DPO data
- Construct a current TRL DPO trainer

In [ ]:
%pip install -q transformers==5.14.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 trl==1.10.0 sentence-transformers==5.7.0 pandas matplotlib scikit-learn requests jsonschema

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## DPO in plain language

For the same prompt we have:
- `chosen`: preferred answer
- `rejected`: inferior answer

DPO directly trains the policy to prefer chosen over rejected relative to a reference model.
It avoids the separate reward-model + online RL loop of classic RLHF.

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files=str(DATA/"network_preferences.jsonl"), split="train")
print(ds)
print(ds[0])

In [ ]:
# Teaching-scale trainer configuration. GPU required for actual training.
from trl import DPOConfig, DPOTrainer

cfg = DPOConfig(
    output_dir=str(ROOT/"artifacts"/"network-dpo"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    max_steps=10,
    report_to="none",
)
print(cfg)

# trainer = DPOTrainer(
#     model="Qwen/Qwen3-0.6B",
#     args=cfg,
#     train_dataset=ds,
# )
# trainer.train()

## When DPO is useful for a NOC assistant

Good:
- consistent preferences with high-quality comparisons,
- operational style and safety behavior,
- evidence-first responses.

Bad:
- trying to teach today's inventory,
- noisy preferences with no rubric,
- using preference optimization before you have a measurable SFT baseline.

### Exercise

Create five chosen/rejected pairs where the only difference is **operational quality**, not verbosity.